In [ ]:
import os, json, time, warnings
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as scipy_signal
warnings.filterwarnings('ignore')

# ── CAMINHOS (mesmo do Passo 2) ──────────────────────────────────
PASTA_RAIZ    = r"C:\Users\vish8\OneDrive\Documentos\TCC\data\scedc-pds"
PASTA_XML     = os.path.join(PASTA_RAIZ, "FDSNstationXML")
PASTA_PROJETO = r"C:\Users\vish8\OneDrive\Documentos\TCC\Trabalho\artefacts"
INVENTARIO    = os.path.join(PASTA_PROJETO, "data", "inventario_dados.csv")
PASTA_WINDOWS = os.path.join(PASTA_PROJETO, "data", "windows")
PASTA_RESULTS = os.path.join(PASTA_PROJETO, "results")
os.makedirs(PASTA_RESULTS, exist_ok=True)
os.makedirs(os.path.join(PASTA_PROJETO, "figures"), exist_ok=True)

# ── PARÂMETROS DO PIPELINE (idênticos ao Passo 2) ────────────────
SR_ALVO     = 40.0
CANAL_ALVO  = "BHZ"
OUTPUT_RESP = "VEL"
PRE_FILT    = (0.5, 1.0, 18.0, 20.0)
WATER_LEVEL = 60
FREQ_MIN    = 0.5
FREQ_MAX    = 15.0

# ── PARÂMETROS STA/LTA (valores iniciais — serão otimizados) ─────
STA_SEG     = 1.0    # duração da janela curta (segundos)
LTA_SEG     = 10.0   # duração da janela longa (segundos)
THRESH_ON   = 3.0    # threshold para disparar detecção
THRESH_OFF  = 1.5    # threshold para encerrar detecção

print("Configuração carregada")
print(f"   STA        : {STA_SEG}s = {int(STA_SEG*SR_ALVO)} amostras")
print(f"   LTA        : {LTA_SEG}s = {int(LTA_SEG*SR_ALVO)} amostras")
print(f"   Threshold  : ON={THRESH_ON} | OFF={THRESH_OFF}")


ModuleNotFoundError: No module named 'config'

Implementação

STA(t) = média( |x(i)|² )  para i ∈ [t - nSTA, t]

LTA(t) = média( |x(i)|² )  para i ∈ [t - nLTA, t]

CF(t)  = STA(t) / LTA(t)   ← Characteristic Function


def sta_lta_manual(sinal, sr,sta_seg, lta_seg):
    """
        Implementação manual do STA/LTA para detecção de eventos sísmicos.
        
        Calcula caracateristicas da função para cada amostra do sinal
        
        Parametros:
        - sinal: array do sinal sísmico
        - sr: taxa de amostragem do sinal
        - sta_seg: duração da janela curta (em segundos)
        - lta_seg: duração da janela longa (em segundos)
        
        Retorna:
        cf = array com a caracaterisca da função (mesmo tamanho do sinal)
    """
    nSTA = int(sta_seg * sr)  # número de amostras na janela curta
    nLTA = int(lta_seg * sr)  # número de amostras na janela longa 
    #Energia instantanea 
    energia = sinal**2
    
    cf = np.zeros(len(sinal))  # array para armazenar os valores da função de característica
    
    #Calculo sta e lta 
    for t in range(nLTA, len(sinal)):
        sta = np.mean(energia[t - nSTA:t])  # energia média na janela curta
        lta = np.mean(energia[t - nLTA:t])  # energia média na janela longa
        if lta > 0:
            cf[t] = sta / lta  # função de característica é a razão entre STA e LTA
    
    return cf

In [ ]:
def sta_lta_eficiente(sinal, sr, sta_seg, lta_seg):
    """
    Versão eficiente usando convolução (muito mais rápida para sinais longos).
    Matematicamente equivalente à versão manual.
    """
    nSTA = int(sta_seg * sr)
    nLTA = int(lta_seg * sr)
    
    energia = sinal ** 2
    
    # Média móvel usando convolução
    kernel_sta = np.ones(nSTA) / nSTA
    kernel_lta = np.ones(nLTA) / nLTA
    
    sta = np.convolve(energia, kernel_sta, mode='full')[:len(sinal)]
    lta = np.convolve(energia, kernel_lta, mode='full')[:len(sinal)]
    
    # Evita divisão por zero
    cf = np.where(lta > 1e-20, sta / lta, 0.0)
    
    return cf

def detectar_eventos(cf, thresh_on, thresh_off, sr, min_duracao_seg=1.0):
    """
    Aplica threshold na CF para detectar eventos.
    
    Lógica:
    - ON  : CF > thresh_on  → inicia evento
    - OFF : CF < thresh_off → encerra evento
    
    Retorna lista de (t_inicio, t_fim) em segundos.
    """
    eventos = []
    min_amostras = int(min_duracao_seg * sr)
    dentro = False
    t_inicio = 0
     
    for t,val in enumerate(cf): #Enumerate = range(len(cf))
        if not dentro and val > thresh_on:
            dentro = True
            t_inicio = t
        elif dentro and val < thresh_off:
            dentro = False
            t_fim = t
            if (t_fim - t_inicio) >= min_amostras:
                eventos.append((t_inicio / sr, t_fim / sr))
    return eventos

In [ ]:
import pandas as pd
from obspy import read, read_inventory

def encontrar_xml(rede, estacao, pasta_xml):
    for root, _, files in os.walk(pasta_xml):
        for f in files:
            if f.endswith('.xml') and rede in f and estacao in f:
                return os.path.join(root, f)
    return None

def processar_trace_raw(tr_input, caminho_xml):
    """Pipeline completo — retorna dados processados."""
    try:
        tr = tr_input.copy()
        if tr.stats.sampling_rate != SR_ALVO:
            tr.resample(SR_ALVO)
        tr.detrend('linear')
        tr.detrend('demean')
        tr.taper(max_percentage=0.05, type='cosine')
        inv = read_inventory(caminho_xml)
        tr.remove_response(inventory=inv, output=OUTPUT_RESP,
                           pre_filt=PRE_FILT, water_level=WATER_LEVEL)
        tr.filter('bandpass', freqmin=FREQ_MIN, freqmax=FREQ_MAX, zerophase=True)
        dados = tr.data.astype(np.float32)
        if dados.std() < 1e-15:
            return None
        return dados
    except:
        return None

# Carregar inventário e pegar o melhor arquivo
df_inv = pd.read_csv(INVENTARIO)
df_uso = df_inv[
    (df_inv['xml_disponivel'] == True) &
    (df_inv['razao_var'] > 10)
].copy().reset_index(drop=True)

row = df_uso.loc[df_uso['razao_var'].idxmax()]
print(f"Arquivo: {row['arquivo']} | razão_var: {row['razao_var']:.0f}x | t_evento: {row['t_evento_est']}s")

st  = read(row['caminho'])
xml = encontrar_xml(row['rede'], row['estacao'], PASTA_XML)

# Usar primeiro trace BHZ disponível
tr_bhz = next((tr for tr in st if tr.stats.channel.endswith(CANAL_ALVO)), None)
dados  = processar_trace_raw(tr_bhz, xml) if tr_bhz else None

if dados is not None:
    tempo = np.arange(len(dados)) / SR_ALVO
    
    # Calcular CF
    cf = sta_lta_eficiente(dados, SR_ALVO, STA_SEG, LTA_SEG)
    
    # Detectar eventos
    eventos_detectados = detectar_eventos(cf, THRESH_ON, THRESH_OFF, SR_ALVO)
    
    # ── Plot ──────────────────────────────────────────────────────
    fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
    
    # 1) Sinal processado
    ax = axes[0]
    ax.plot(tempo, dados, 'steelblue', lw=0.6)
    ax.axvline(row['t_evento_est'], color='red', ls='--', lw=1.5,
               label=f"t_real = {row['t_evento_est']}s")
    for t0, t1 in eventos_detectados:
        ax.axvspan(t0, t1, alpha=0.25, color='orange', label='STA/LTA detectou')
    ax.set_ylabel('Velocidade (m/s)')
    ax.set_title(f'Sinal processado — {row["rede"]}.{row["estacao"]}.{CANAL_ALVO}')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    # 2) Characteristic Function
    ax = axes[1]
    ax.plot(tempo, cf, 'darkorange', lw=1.0)
    ax.axhline(THRESH_ON,  color='red',   ls='--', lw=1.2,
               label=f'Thresh ON = {THRESH_ON}')
    ax.axhline(THRESH_OFF, color='green', ls='--', lw=1.2,
               label=f'Thresh OFF = {THRESH_OFF}')
    ax.axvline(row['t_evento_est'], color='red', ls=':', lw=1.2, alpha=0.7)
    ax.set_ylabel('CF = STA/LTA')
    ax.set_title('Characteristic Function STA/LTA')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, min(cf.max() * 1.1, 30)])
    
    # 3) Espectrograma
    ax   = axes[2]
    NFFT = int(2 * SR_ALVO)
    f_s, t_s, Sxx = scipy_signal.spectrogram(dados, fs=SR_ALVO,
                                               nperseg=NFFT, noverlap=NFFT//2)
    im = ax.pcolormesh(t_s, f_s, 10*np.log10(Sxx+1e-20),
                       shading='gouraud', cmap='inferno')
    ax.set_ylim([0, 20])
    ax.axvline(row['t_evento_est'], color='white', ls='--', lw=1.5, alpha=0.8)
    ax.set_ylabel('Frequência (Hz)')
    ax.set_xlabel('Tempo (s)')
    ax.set_title('Espectrograma')
    plt.colorbar(im, ax=ax, label='dB')
    
    plt.suptitle(f'STA/LTA — {row["arquivo"]} | STA={STA_SEG}s LTA={LTA_SEG}s thr={THRESH_ON}',
                 fontsize=11)
    plt.tight_layout()
    fig_path = os.path.join(PASTA_PROJETO, "figures", "passo3_staltalvizualizacao.png")
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nEventos detectados pelo STA/LTA: {len(eventos_detectados)}")
    for i, (t0, t1) in enumerate(eventos_detectados):
        print(f"  Evento {i+1}: {t0:.1f}s → {t1:.1f}s (duração: {t1-t0:.1f}s)")
    print(f"\nEvento real em: {row['t_evento_est']}s")


Agora rodamos o STA/LTA em todos os 22 arquivos e calculamos as métricas.

### Como definimos ground truth
O `t_evento_est` do inventário (Passo 1) é o nosso ground truth.
Um arquivo é considerado **detecção correta (VP)** se o STA/LTA disparou
dentro de uma janela de ±15s do t_evento_est.

### Métricas calculadas
- **Precisão**: dos alarmes disparados, quantos eram eventos reais?
- **Recall**: dos eventos reais, quantos foram detectados?
- **F1-score**: média harmônica entre precisão e recall
- **AUC-ROC**: variando o threshold, qual é o trade-off completo?


In [ ]:
from sklearn.metrics import (precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve)

TOLERANCIA_SEG = 15.0   # janela de tolerância para considerar detecção correta

resultados = []
cf_scores  = []   # score contínuo para curva ROC
labels     = []   # 0=ruído, 1=evento (ground truth)

print(f"Avaliando STA/LTA em {len(df_uso)} arquivos...")
print(f"Parâmetros: STA={STA_SEG}s | LTA={LTA_SEG}s | thr={THRESH_ON}/{THRESH_OFF}")
print(f"Tolerância: ±{TOLERANCIA_SEG}s")
print("-" * 70)

tempo_total_inferencia = 0.0
n_amostras_total       = 0

for _, row in df_uso.iterrows():
    try:
        st  = read(row['caminho'])
        xml = encontrar_xml(row['rede'], row['estacao'], PASTA_XML)
        if xml is None:
            continue
        
        tr_bhz = next((tr for tr in st if tr.stats.channel.endswith(CANAL_ALVO)), None)
        if tr_bhz is None:
            continue
        
        dados = processar_trace_raw(tr_bhz, xml)
        if dados is None:
            continue
        
        # Medir tempo de inferência
        t0_inf = time.perf_counter()
        cf = sta_lta_eficiente(dados, SR_ALVO, STA_SEG, LTA_SEG)
        t1_inf = time.perf_counter()
        
        tempo_total_inferencia += (t1_inf - t0_inf)
        n_amostras_total       += len(dados)
        
        # Detectar eventos
        eventos = detectar_eventos(cf, THRESH_ON, THRESH_OFF, SR_ALVO)
        
        # Ground truth: evento real em t_evento_est
        t_real    = row['t_evento_est']
        detectado = False
        
        for t_ini, t_fim in eventos:
            # Considera detecção correta se o alarme ocorreu próximo do evento real
            if abs(t_ini - t_real) <= TOLERANCIA_SEG or (t_ini <= t_real <= t_fim):
                detectado = True
                break
        
        # Score contínuo: CF máxima na janela do evento (para curva ROC)
        idx_ini = max(0, int((t_real - TOLERANCIA_SEG) * SR_ALVO))
        idx_fim = min(len(cf), int((t_real + TOLERANCIA_SEG) * SR_ALVO))
        score_evento = float(cf[idx_ini:idx_fim].max()) if idx_fim > idx_ini else 0.0
        
        # Score de ruído: CF máxima fora da janela do evento
        cf_ruido    = np.concatenate([cf[:idx_ini], cf[idx_fim:]])
        score_ruido = float(cf_ruido.max()) if len(cf_ruido) > 0 else 0.0
        
        cf_scores.extend([score_ruido, score_evento])
        labels.extend([0, 1])
        
        resultados.append({
            'arquivo'  : row['arquivo'],
            'razao_var': row['razao_var'],
            't_real'   : t_real,
            'detectado': detectado,
            'n_alarmes': len(eventos),
            'score'    : score_evento,
        })
        
        status = "VP" if detectado else " FN"
        print(f"  {status} {row['arquivo']:25s}  CF_max={score_evento:6.2f}  "
              f"alarmes={len(eventos)}  t_real={t_real:.0f}s")
    
    except Exception as e:
        print(f"  {e}")

print()
print("=" * 70)

# ── Métricas finais ───────────────────────────────────────────────
df_res = pd.DataFrame(resultados)

y_true = df_res['detectado'].astype(int).values
y_pred = df_res['detectado'].astype(int).values  # threshold fixo

vp = y_true.sum()
fn = len(y_true) - vp

precisao = vp / max(vp, 1)   # sem falsos positivos por arquivo (1 evento por arquivo)
recall   = vp / max(len(y_true), 1)
f1       = 2 * precisao * recall / max(precisao + recall, 1e-10)

# AUC com scores contínuos
try:
    auc = roc_auc_score(labels, cf_scores)
except:
    auc = 0.0

# Tempo de inferência
dur_media_s   = n_amostras_total / SR_ALVO / len(df_uso)
ms_por_seg    = (tempo_total_inferencia / len(df_uso)) * 1000

print(f"RESULTADOS STA/LTA")
print(f"  Arquivos avaliados : {len(df_res)}")
print(f"  Detecções corretas : {vp} / {len(df_res)}")
print(f"  Precisão           : {precisao:.4f}")
print(f"  Recall             : {recall:.4f}")
print(f"  F1-score           : {f1:.4f}")
print(f"  AUC-ROC            : {auc:.4f}")
print(f"  Tempo inferência   : {ms_por_seg:.2f} ms/arquivo")
print(f"  Tamanho do modelo  : N/A (sem modelo, é um algoritmo)")


O STA/LTA tem 4 parâmetros: `sta`, `lta`, `thresh_on`, `thresh_off`.
Vamos variar cada um e ver o impacto no F1.


1. Encontrar os melhores parâmetros para o baseline
2. Mostrar no paper que o STA/LTA é sensível à escolha de parâmetros
   (enquanto o autoencoder aprende automaticamente)


In [ ]:
def avaliar_parametros(df_uso, sta_seg, lta_seg, thresh_on, thresh_off=None):
    """Avalia STA/LTA com um conjunto de parâmetros e retorna F1 e AUC."""
    if thresh_off is None:
        thresh_off = thresh_on * 0.5
    
    scores, labs = [], []
    deteccoes    = []
    
    for _, row in df_uso.iterrows():
        try:
            st  = read(row['caminho'])
            xml = encontrar_xml(row['rede'], row['estacao'], PASTA_XML)
            if xml is None: continue
            tr_bhz = next((tr for tr in st if tr.stats.channel.endswith(CANAL_ALVO)), None)
            if tr_bhz is None: continue
            dados = processar_trace_raw(tr_bhz, xml)
            if dados is None: continue
            
            cf      = sta_lta_eficiente(dados, SR_ALVO, sta_seg, lta_seg)
            eventos = detectar_eventos(cf, thresh_on, thresh_off, SR_ALVO)
            t_real  = row['t_evento_est']
            
            detectado = any(
                abs(t0 - t_real) <= TOLERANCIA_SEG or (t0 <= t_real <= t1)
                for t0, t1 in eventos
            )
            deteccoes.append(int(detectado))
            
            idx_ini = max(0, int((t_real - TOLERANCIA_SEG) * SR_ALVO))
            idx_fim = min(len(cf), int((t_real + TOLERANCIA_SEG) * SR_ALVO))
            scores.append(float(cf[idx_ini:idx_fim].max()) if idx_fim > idx_ini else 0.0)
            labs.append(1)
            cf_r = np.concatenate([cf[:idx_ini], cf[idx_fim:]])
            scores.append(float(cf_r.max()) if len(cf_r)>0 else 0.0)
            labs.append(0)
        except:
            continue
    
    if not deteccoes:
        return 0.0, 0.0
    
    recall = sum(deteccoes) / len(deteccoes)
    f1     = recall  # precisão = 1 (1 evento por arquivo, sem FP por arquivo)
    try:
        auc = roc_auc_score(labs, scores)
    except:
        auc = 0.0
    return f1, auc


print("Análise de sensibilidade — variando parâmetros")
print("=" * 60)

# ── Variar STA ────────────────────────────────────────────────────
stas    = [0.5, 1.0, 2.0, 3.0, 5.0]
f1_sta  = []
auc_sta = []
print("\nVariando STA (LTA=10s, thr=3.0):")
for sta in stas:
    f1, auc = avaliar_parametros(df_uso, sta, 10.0, 3.0)
    f1_sta.append(f1); auc_sta.append(auc)
    print(f"  STA={sta:4.1f}s  F1={f1:.3f}  AUC={auc:.3f}")

# ── Variar LTA ────────────────────────────────────────────────────
ltas    = [5.0, 10.0, 15.0, 20.0, 30.0]
f1_lta  = []
auc_lta = []
print("\nVariando LTA (STA=1s, thr=3.0):")
for lta in ltas:
    f1, auc = avaliar_parametros(df_uso, 1.0, lta, 3.0)
    f1_lta.append(f1); auc_lta.append(auc)
    print(f"  LTA={lta:5.1f}s  F1={f1:.3f}  AUC={auc:.3f}")

# ── Variar threshold ──────────────────────────────────────────────
thrs    = [1.5, 2.0, 3.0, 4.0, 5.0, 7.0, 10.0]
f1_thr  = []
auc_thr = []
print("\nVariando threshold (STA=1s, LTA=10s):")
for thr in thrs:
    f1, auc = avaliar_parametros(df_uso, 1.0, 10.0, thr)
    f1_thr.append(f1); auc_thr.append(auc)
    print(f"  thr={thr:5.1f}  F1={f1:.3f}  AUC={auc:.3f}")

# ── Plot sensibilidade ────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(stas, f1_sta, 'steelblue', marker='o', lw=2)
axes[0].set_xlabel('Duração STA (s)')
axes[0].set_ylabel('F1-score')
axes[0].set_title('Sensibilidade ao STA')
axes[0].grid(True, alpha=0.3)

axes[1].plot(ltas, f1_lta, 'darkorange', marker='o', lw=2)
axes[1].set_xlabel('Duração LTA (s)')
axes[1].set_ylabel('F1-score')
axes[1].set_title('Sensibilidade ao LTA')
axes[1].grid(True, alpha=0.3)

axes[2].plot(thrs, f1_thr, 'crimson', marker='o', lw=2)
axes[2].set_xlabel('Threshold ON')
axes[2].set_ylabel('F1-score')
axes[2].set_title('Sensibilidade ao Threshold')
axes[2].grid(True, alpha=0.3)

plt.suptitle('STA/LTA — Análise de Sensibilidade dos Parâmetros', fontsize=11)
plt.tight_layout()
fig_path = os.path.join(PASTA_PROJETO, "figures", "passo3_sensibilidade.png")
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()

# Melhores parâmetros encontrados
best_sta = stas[np.argmax(f1_sta)]
best_lta = ltas[np.argmax(f1_lta)]
best_thr = thrs[np.argmax(f1_thr)]
print(f"\nMelhores parâmetros encontrados:")
print(f"  STA = {best_sta}s | LTA = {best_lta}s | threshold = {best_thr}")


In [ ]:
# Recalcular com melhores parâmetros para a curva ROC final
scores_roc, labels_roc = [], []

for _, row in df_uso.iterrows():
    try:
        st  = read(row['caminho'])
        xml = encontrar_xml(row['rede'], row['estacao'], PASTA_XML)
        if xml is None: continue
        tr_bhz = next((tr for tr in st if tr.stats.channel.endswith(CANAL_ALVO)), None)
        if tr_bhz is None: continue
        dados = processar_trace_raw(tr_bhz, xml)
        if dados is None: continue
        
        cf     = sta_lta_eficiente(dados, SR_ALVO, best_sta, best_lta)
        t_real = row['t_evento_est']
        
        idx_ini = max(0, int((t_real - TOLERANCIA_SEG) * SR_ALVO))
        idx_fim = min(len(cf), int((t_real + TOLERANCIA_SEG) * SR_ALVO))
        
        scores_roc.append(float(cf[idx_ini:idx_fim].max()) if idx_fim > idx_ini else 0.0)
        labels_roc.append(1)
        
        cf_r = np.concatenate([cf[:idx_ini], cf[idx_fim:]])
        scores_roc.append(float(cf_r.max()) if len(cf_r) > 0 else 0.0)
        labels_roc.append(0)
    except:
        continue

# Curva ROC
fpr, tpr, thresholds = roc_curve(labels_roc, scores_roc)
auc_final = roc_auc_score(labels_roc, scores_roc)

# F1 para cada threshold da ROC
f1_roc = []
for thr in thresholds:
    y_pred_thr = (np.array(scores_roc) >= thr).astype(int)
    f1_roc.append(f1_score(labels_roc, y_pred_thr, zero_division=0))

idx_best_f1  = np.argmax(f1_roc)
best_f1_val  = f1_roc[idx_best_f1]
best_thr_roc = thresholds[idx_best_f1]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ROC
ax = axes[0]
ax.plot(fpr, tpr, 'steelblue', lw=2, label=f'STA/LTA (AUC = {auc_final:.3f})')
ax.plot([0,1],[0,1],'k--', lw=1, alpha=0.5, label='Aleatório (AUC = 0.500)')
ax.scatter(fpr[idx_best_f1], tpr[idx_best_f1], color='red', s=100, zorder=5,
           label=f'Melhor F1 = {best_f1_val:.3f} (thr={best_thr_roc:.2f})')
ax.set_xlabel('Taxa de Falso Positivo')
ax.set_ylabel('Taxa de Verdadeiro Positivo (Recall)')
ax.set_title('Curva ROC — STA/LTA')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# F1 por threshold
ax = axes[1]
ax.plot(thresholds, f1_roc, 'crimson', lw=2)
ax.axvline(best_thr_roc, color='red', ls='--', lw=1.5,
           label=f'Threshold ótimo = {best_thr_roc:.2f}')
ax.set_xlabel('Threshold')
ax.set_ylabel('F1-score')
ax.set_title('F1 por Threshold — STA/LTA')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle('STA/LTA — Curva ROC e Threshold Ótimo', fontsize=11)
plt.tight_layout()
fig_path = os.path.join(PASTA_PROJETO, "figures", "passo3_roc.png")
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"AUC-ROC final : {auc_final:.4f}")
print(f"Melhor F1     : {best_f1_val:.4f} (threshold = {best_thr_roc:.2f})")


In [ ]:
# Avaliação final com melhores parâmetros
f1_final, auc_f = avaliar_parametros(df_uso, best_sta, best_lta, best_thr_roc)

# Tempo de inferência por segundo de sinal
ms_por_s_sinal = (tempo_total_inferencia / n_amostras_total) * SR_ALVO * 1000

print("=" * 65)
print("PRIMEIRA LINHA DA TABELA COMPARATIVA — STA/LTA")
print("=" * 65)
print(f"  Parâmetros   : STA={best_sta}s | LTA={best_lta}s | thr={best_thr_roc:.2f}")
print(f"  F1-score     : {f1_final:.4f}")
print(f"  AUC-ROC      : {auc_final:.4f}")
print(f"  Tamanho      : N/A (algoritmo, não modelo)")
print(f"  Inferência   : {ms_por_s_sinal:.3f} ms por segundo de sinal")
print(f"  Quantização  : N/A")
print()
print("Tabela parcial do paper:")
print(f"  {'Método':<15} {'F1':>8} {'AUC':>8} {'Tamanho':>10} {'Inf.(ms)':>12}")
print(f"  {'-'*15} {'-'*8} {'-'*8} {'-'*10} {'-'*12}")
print(f"  {'STA/LTA':<15} {f1_final:>8.4f} {auc_final:>8.4f} {'N/A':>10} {ms_por_s_sinal:>12.3f}")
print(f"  {'Dense AE':<15} {'—':>8} {'—':>8} {'—':>10} {'—':>12}")
print(f"  {'CNN 1D AE':<15} {'—':>8} {'—':>8} {'—':>10} {'—':>12}")
print(f"  {'LSTM AE':<15} {'—':>8} {'—':>8} {'—':>10} {'—':>12}")

# Salvar resultado
resultado_staltalvta = {
    "metodo"      : "STA/LTA",
    "parametros"  : {"sta_seg": best_sta, "lta_seg": best_lta, "threshold": float(best_thr_roc)},
    "f1"          : float(f1_final),
    "auc"         : float(auc_final),
    "tamanho_kb"  : None,
    "inferencia_ms": float(ms_por_s_sinal),
    "quantizado_f1": None,
    "quantizado_ms": None,
}

result_path = os.path.join(PASTA_RESULTS, "resultado_staltalvta.json")
with open(result_path, 'w') as f:
    json.dump(resultado_staltalvta, f, indent=2)

print(f"\nResultado salvo: {result_path}")
